In [20]:
from qiskit import QuantumCircuit
from qforte.system import system_factory
from qforte.evolution.trotter import trotter_evolve, hartree_fock
from qforte.qiskit_api.translators import qforte_to_qiskit
from qforte.qiskit_api.dispatchers import AerDispatcher

geom = [
    ('H', (0., 0., 1.0)), 
    ('H', (0., 0., 2.0))
    ]

mol = system_factory(
    build_type='psi4',
    mol_geometry=geom,
    basis='sto-3g',
    df_icut=1.0e-6)

#number of krylov basis states to generate
krylov_basis_size = 3
#timestep size (seconds)
dt = 0.1

#computational backend to use with Qiskit
aer_dispatcher = AerDispatcher();
#number of samples
shots = 10000

#construct the hartree fock and zero states for H2
hf_state = hartree_fock(mol, QuantumCircuit(4))
zero_state = QuantumCircuit(4)

#create a circuit to prepare the GHZ state on 4 qubits
ghz_circ = QuantumCircuit(4)
ghz_circ.h(0)
ghz_circ.cx(0, 1)

#prepare the GHZ state by applying the GHZ circuit to the zero state
ghz_state = zero_state.compose(ghz_circ)

#construct the 1st order trotterized evolution circuit for H2
qf_trotter_circ, qf_trotter_computer = trotter_evolve(mol, dt)
qiskit_trotter_circ = qforte_to_qiskit(qf_trotter_circ, qf_trotter_computer.get_nqubit())

#get the first krylov basis state by applying the trotterized evolution circuit to both the hartree fock state and the GHZ state
hf_krylov_states = [hf_state.compose(qiskit_trotter_circ)]
ghz_krylov_states = [ghz_state.compose(qiskit_trotter_circ)]

#get the rest of the krylov basis states by repeatedly applying the trotterized evolution circuit to both the hartree fock state and the GHZ state
for i in range(krylov_basis_size - 1):
    hf_krylov_states.append(hf_krylov_states[-1].compose(qiskit_trotter_circ))
    ghz_krylov_states.append(ghz_krylov_states[-1].compose(qiskit_trotter_circ))

#apply the inverse of the GHZ circuit to all the GHZ-based krylov states to map the GHZ state back to the zero state
ghz_krylov_states = [state.compose(ghz_circ.inverse()) for state in ghz_krylov_states]

#append measurements to all the circuits
for state in hf_krylov_states:
    state.measure_all()
for state in ghz_krylov_states:
    state.measure_all()
    
#dispatch the circuits to aer for sampling
aer_dispatcher = AerDispatcher(); shots = 10000
hf_krylov_results = aer_dispatcher.dispatch_sampler(hf_krylov_states, shots)
ghz_krylov_results = aer_dispatcher.dispatch_sampler(ghz_krylov_states, shots)

#extract the sample counts from Qiskit's data structures
hf_krylov_samples = [res.data.meas.get_counts() for res in hf_krylov_results]
ghz_krylov_samples = [res.data.meas.get_counts() for res in ghz_krylov_results]

#calculate F1 for each krylov state by taking the count of the '0011' bitstring and dividing by the total number of shots
F1 = [sample["0011"]/shots for sample in hf_krylov_samples]
print(f"\nF1 values for {krylov_basis_size} krylov states with dt = {dt} seconds: {F1}")

#calculate F2 for each krylov state by taking the count of the '0000' bitstring (mapped to the GHZ state) and dividing by the total number of shots
F2 = [sample["0000"]/shots for sample in ghz_krylov_samples]
print(f"\nF2 values for {krylov_basis_size} krylov states with dt = {dt} seconds: {F2}")

#calculate r values by taking the square root of F1 for each krylov state
r = [f1_val**0.5 for f1_val in F1]
print(f"\nr values for {krylov_basis_size} krylov states with dt = {dt} seconds: {r}")

 ==> Psi4 geometry <==
-------------------------
0  1
H  0.0  0.0  1.0
H  0.0  0.0  2.0
symmetry c1
units angstrom

F1 values for 3 krylov states with dt = 0.1 seconds: [0.9997, 0.9991, 0.9965]

F2 values for 3 krylov states with dt = 0.1 seconds: [0.9937, 0.9754, 0.9443]

r values for 3 krylov states with dt = 0.1 seconds: [0.9998499887483122, 0.9995498987044118, 0.9982484660644363]
